# 18. Multi-Stage Training, RLHF, PPO, and GRPO

## Problem

一个 DeepSeek-like 模型为什么不是“训一次就结束”？从最开始的 base model，到后面的 SFT、reward model、RLHF、PPO、GRPO，这些阶段各自解决什么问题，它们为什么要按这个顺序出现？

## Dependencies

建议先读完 pretraining、SFT、reward model 那几节，再进入这一节。因为这里不再只讲单个模块，而是讲整个训练主线怎样一层层叠起来。


## Why strong models are built in stages

现代大模型的训练主线通常不是一步到位，而是分阶段塑形。原因很简单：**不同阶段解决的不是同一个问题。**

- pretraining 解决的是“先把语言和世界模式压进参数”。
- continued training 解决的是“把某些关键能力继续拉高”。
- SFT 解决的是“让模型更像一个会配合任务的助手”。
- reward model 解决的是“在人类更喜欢哪种回答这件事上，造出一个可学习信号”。
- RLHF / PPO / GRPO / DPO 解决的是“怎样把偏好信号真正压回 policy 本身”。

如果把这些阶段混成一锅看，就很容易误以为：

- RLHF 是模型获得知识的地方
- SFT 是主要能力来源
- PPO 和 GRPO 只是换名字

这些理解都不准。真正更准确的视角是：**先做能力底座，再做能力偏置，再做行为塑形，再做偏好优化。**


## The full training chain

可以先把整条链写成一句话：

$$
\text{Pretraining} \rightarrow \text{Continued Training} \rightarrow \text{SFT} \rightarrow \text{Reward Model} \rightarrow \text{PPO / DPO / GRPO}
$$

现在把它逐段拆开。

### 1. Pretraining

这一阶段的输入是大规模通用语料：网页、代码、文档、书籍、论文、讨论内容等。目标是 next-token prediction。

它的产出不是“好助手”，而是一个强 base model，也就是：

- 会续写
- 有广义语言能力
- 有知识共现记忆
- 有一部分通用推理模板

### 2. Continued training

很多项目不会在预训练后立刻做 SFT，而是先用更集中、更高价值的数据做第二轮强化。例如：

- 数学
- 代码
- 推理
- 长上下文文本
- 某些专业领域语料

这一阶段不是换训练范式，很多时候仍然是 next-token prediction，只是数据分布更聚焦。它的任务是：**把基础能力往关键方向继续推高。**

### 3. SFT

SFT 把模型从“会续写文本”继续推向“更会按指令回答”。它更像给模型一个助手行为骨架。

### 4. Reward model

reward model 不直接生成答案，而是学会在两个答案里判断哪一个更符合偏好。它的作用是把 preference data 压缩成一个可训练分数信号。

### 5. Policy optimization

到了这一步，真正的问题变成：怎样让主模型更稳定地偏向高偏好回答？

这就是 PPO、GRPO、DPO 一类方法登场的位置。


In [ ]:
# 先把整条训练链做成一个结构化视图。
# 这不是训练脚本，而是帮助建立“每一阶段输入什么、输出什么”的脑图。

training_stages = [
    {
        'stage': 'pretraining',
        'input_data': 'web + code + docs + books + papers',
        'objective': 'next-token prediction',
        'output': 'general language backbone',
    },
    {
        'stage': 'continued_training',
        'input_data': 'math / code / reasoning / domain-focused corpora',
        'objective': 'continue next-token prediction on targeted distributions',
        'output': 'domain-strengthened backbone',
    },
    {
        'stage': 'sft',
        'input_data': 'instruction-response pairs',
        'objective': 'assistant-style supervised fine-tuning',
        'output': 'instruction-following checkpoint',
    },
    {
        'stage': 'reward_model',
        'input_data': 'chosen-rejected preference pairs',
        'objective': 'fit preference scores',
        'output': 'reward scorer',
    },
    {
        'stage': 'policy_optimization',
        'input_data': 'policy rollouts + reward / preference signal',
        'objective': 'shift policy toward better-ranked answers',
        'output': 'preference-aligned policy',
    },
]

for item in training_stages:
    print(item['stage'])
    print('  input_data =', item['input_data'])
    print('  objective  =', item['objective'])
    print('  output     =', item['output'])


## Why continued training matters so much

continued training 很容易被讲轻，好像只是“预训练的尾巴”。实际上它常常是实际项目里非常关键的一层。

原因在于：

- 通用预训练分布太广，不可能把每类能力都拉到同样强。
- 某些项目特别关心代码、数学、工具使用、推理或长文档理解。
- 如果直接从通用预训练跳到 SFT，模型可能会在助手行为上更配合，但底层能力还没补足。

所以 continued training 的价值可以概括成一句话：**在还不急着塑造行为之前，先把最重要的底层能力再往上推一截。**


In [ ]:
import numpy as np

# 一个 toy 版本的“训练预算分配”示例。
# 数字没有现实意义，只是为了帮助理解不同阶段在总项目中的相对投入。

stage_token_budget = {
    'pretraining': 1000,
    'continued_training': 180,
    'sft': 25,
    'reward_model': 8,
    'policy_optimization': 12,
}

total_budget = sum(stage_token_budget.values())
for stage, budget in stage_token_budget.items():
    share = budget / total_budget
    print(f'{stage:20s} share = {share:.4f}')


## Version lineage and why checkpoint tracking matters

一旦训练变成多阶段，你就不能再把“模型”当成一个单一对象看。更准确地说，项目里会有一串版本：

- base pretrained checkpoint
- continued training checkpoint
- SFT checkpoint
- reward model checkpoint
- PPO / GRPO / DPO 之后的 policy checkpoint

如果不追踪这条 lineage，后面很多问题会说不清：

- 某项能力是在哪一阶段增强的？
- 某种行为退化是在哪一阶段引入的？
- 一个新版本是继续在谁的基础上训练出来的？

这也是为什么真正做项目时，checkpoint 管理不是可有可无的附属工作。


In [ ]:
# 用一个最小 lineage 列表模拟版本流转。

checkpoint_lineage = [
    'v1_base_pretrained',
    'v2_continued_training',
    'v3_sft',
    'v4_reward_model',
    'v5_ppo_or_grpo_policy',
]

for idx, ckpt in enumerate(checkpoint_lineage, start=1):
    print(f'{idx}. {ckpt}')


## PPO in one picture

PPO 的核心思想，不是“让高分答案更高分”这么简单，而是：

- 希望 policy 朝更好回答更新
- 但又不希望一步更新得太猛

一个常见写法是：

$$
r_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{old}(a_t \mid s_t)}
$$

$$
L^{PPO} = \mathbb{E}\Big[\min\big(r_t A_t, \operatorname{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t\big)\Big]
$$

这里：

- $r_t$ 是新旧策略概率比。
- $A_t$ 是 advantage，表示这个动作相对基线到底更好还是更差。
- clip 的作用是限制更新幅度，防止策略一下子跑太远。

如果只看直觉，PPO 在做的就是：**朝更好回答更新，但给更新步幅装一个护栏。**


In [ ]:
# 一个最小 PPO 计算例子。
# advantage > 0 说明这次回答相对更值得被鼓励。
# advantage < 0 说明这次回答应该被压低概率。

old_logprob = np.array([-1.20, -0.80, -1.50])
new_logprob = np.array([-1.00, -0.95, -1.10])
advantage = np.array([0.8, -0.4, 0.5])
clip_eps = 0.2

ratio = np.exp(new_logprob - old_logprob)
clipped_ratio = np.clip(ratio, 1 - clip_eps, 1 + clip_eps)
ppo_objective = np.minimum(ratio * advantage, clipped_ratio * advantage)

print('ratio =', ratio)
print('clipped_ratio =', clipped_ratio)
print('advantage =', advantage)
print('ppo_objective =', ppo_objective)


## Why KL control shows up in RLHF

实际 RLHF 里，除了 reward model 给出的分数，还常常要加一个约束：**不要让新 policy 偏离 reference model 太远。**

一个常见写法是：

$$
R_{total} = R_{rm} - \beta \cdot KL(\pi_\theta \;\|\; \pi_{ref})
$$

这条式子的工程含义很强：

- $R_{rm}$ 希望模型往高偏好答案走
- $KL$ 则像刹车，避免 policy 为了追高分而偏离得过猛

如果没有这类约束，模型可能会更容易出现 reward hacking、风格漂移、输出不稳定等问题。


In [ ]:
# 一个最小 total reward 直觉例子。

rm_reward = 2.3
kl_penalty = 0.4
beta_kl = 0.2

total_reward = rm_reward - beta_kl * kl_penalty

print('reward model score =', rm_reward)
print('kl penalty =', kl_penalty)
print('total reward after KL control =', total_reward)


## GRPO in one picture

GRPO 的核心直觉更接近：

- 针对同一个 prompt，一次采样出一组回答
- 不一定强依赖完整的 value-function 估计
- 直接利用组内相对比较来构造更新信号

你可以先把它理解成：**不是问“这条回答的绝对奖励是多少”，而是更强调“在同题这一组回答里，它相对别人排第几”。**

一个很常见的 toy 直觉是先做组内标准化：

$$
A_i^{group} = \frac{r_i - \mu_{group}}{\sigma_{group} + \epsilon}
$$

这里：

- $r_i$ 是组内某个回答的奖励
- $\mu_{group}$ 是这一组回答奖励的均值
- $\sigma_{group}$ 是标准差

这样做的意思是：把同题回答放在一起比较，让“相对更好”这件事直接进入更新信号。


In [ ]:
# 一个最小 GRPO 风格例子。
# 同一 prompt 下采样 4 个回答，只看它们组内谁更好。

group_rewards = np.array([0.62, 0.55, 0.91, 0.48])
group_mean = group_rewards.mean()
group_std = group_rewards.std() + 1e-6
group_advantage = (group_rewards - group_mean) / group_std

print('group_rewards =', group_rewards)
print('group_mean =', group_mean)
print('group_std =', group_std)
print('group_relative_advantage =', group_advantage)


## PPO vs GRPO

这两者不该只被记成“两个缩写”。更有用的比较方式是看它们组织优化信号的方法。

**PPO** 更像：

- 明确对比新旧 policy
- 显式使用 ratio 与 clipping
- 很强调 update constraint
- 在经典 RLHF 叙事里更常见

**GRPO** 更像：

- 更强调同题多答
- 更强调组内相对奖励
- 更直接地把“这一组里谁更好”拿来构造优势信号
- 在某些 reasoning / multi-sample setting 里会更自然

所以真正该问的不是“谁更高级”，而是：

- 你的反馈信号是更适合绝对奖励，还是相对比较？
- 你的 rollout 组织方式更像单条样本，还是同题成组样本？
- 你的训练稳定性约束更依赖哪种结构？

## Common mistakes

- 只记阶段顺序，不理解每一轮训练解决的具体问题。
- 把 RLHF 误解成模型获得知识的主要来源。它更主要是在已有能力上调行为分布。
- 忽略 continued training，以为 pretraining 之后就应该直接 SFT。
- 把 PPO 和 GRPO 当成换名字算法，而不去理解它们组织优势信号的差异。
- 不追踪 checkpoint lineage，最后说不清模型是怎么一步步演化出来的。

## Summary

这一节最重要的收获，不是背住一串缩写，而是看清楚：现代强模型通常是**分阶段被做出来的**。先做能力底座，再做定向补强，再做助手行为塑形，最后再做偏好优化。PPO 和 GRPO 都属于最后这一段，但它们组织更新信号的方式并不一样。
